# Point Stimulation on Anchor-Shaped Cells

Per-cell point stimulation: a small DMD spot is placed on each segmented cell at a chosen location
(top / middle / bottom) — same idea as Moritz's EXP_24 (`Moritz_DMD_top-bottom-side-spot_cellpose_EXP_24.ipynb`),
with the downstream analysis from `Analysis_Exp25.ipynb` / `analysis_exp25_v2.ipynb` (per-cell ERK-KTR
ratio and post-stim AUC, compared across stim location).

This notebook ports the experiment to faro / RTMSequence on the Moench microscope.

**Imaging channels**
* miRFP — LifeAct (cell shape)
* mScarlet3 — ERK-KTR (signaling read-out, also used for segmentation)

**Stimulation channel**
* CyanStim (470 nm) via the Mosaic3 DMD

**Two stimulator variants are provided in `faro/stimulation/spot_on_cell.py`:**
1. `StimSpotOnCell` — discrete `top` / `middle` / `bottom`. Reproduces Moritz's `get_shape_percentage()` (same reference points, same defaults: `spot_radius=5`, `height_percentage=0.6`, `offset=-5`).
2. `StimSpotOnCellPolar` — continuous polar parameterization `(stim_angle_deg, stim_radial_fraction)` plus an optional `stim_spot_radius`. Designed as a smooth 2D / 3D parameter surface for Bayesian optimization.

**Stim mode** — `stim_mode="previous"`: the stim mask at frame N is built from frame N-1's segmentation, and the stim fires *before* frame N's image. With our single pulse at frame 5, that means frame 5 is the first post-stim observation — exactly what `Analysis_Exp25_v2`'s `frame >= STIM_FRAME` post-stim window assumes.

### Imports

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import time
from faro.core.data_structures import Channel, StimTreatment, SegmentationMethod
import faro.core.utils as utils
from pprint import pprint
import pandas as pd
import numpy as np

### Microscope: Moench

In [ ]:
from faro.microscope.pertzlab.moench import Moench

mic = Moench()
# The channel group must contain both imaging configs (miRFP, mScarlet3) AND the
# stimulation config (CyanStim). Adjust to whichever group on Moench bundles them.
CHANNEL_GROUP = "TTL_ERK"
mic.mmc.setChannelGroup(CHANNEL_GROUP)

### Experiment configuration

Defaults follow EXP_24 / EXP_25:
* 30 frames at 60 s spacing → 30 min total
* single stim pulse at frame 5 (5 baseline frames first, so `ratio_ERK_norm` analysis works)
* three treatments — `top` / `middle` / `bottom` — distributed across FOVs

In [ ]:
# Acquisition timing
N_FRAMES = 30
TIME_BETWEEN_TIMESTEPS = 60       # seconds between timesteps
TIME_PER_FOV = 4.0                # seconds per FOV (camera + stage + DMD overhead)
STIM_FRAME = 5                    # single-pulse stim at frame 5 (matches Analysis_Exp25)

SLEEP_BEFORE_EXPERIMENT_START_in_H = 0
ADD_STIM_EXPOSURE_GROUP = False
REGULAR_SPACING_BETWEEN_STIMULATIONS = False

# Storage
base_path = "E:\\Alex"
experiment_name = "point_stimulation_test"
path = os.path.join(base_path, experiment_name)

# Imaging channels: LifeAct (cell shape) + ERK-KTR (signaling).
# Channel index 0 -> miRFP, index 1 -> mScarlet3. Segmentation uses index 1 (ERK-KTR).
channels = [
    Channel(config="miRFP",     exposure=300),  # LifeAct
    Channel(config="mScarlet3", exposure=200),  # ERK-KTR
]

# Optocheck (e.g. mCitrine for optoFGFR) — captured at end of run.
channel_optocheck = Channel(config="mCitrine", exposure=200)
optocheck_timepoints = (N_FRAMES - 1,)

# Cell-line / treatment label per FOV. Repeat or extend to match FOV count.
condition = ["optoFGFR"]

# Wellplate layout (None = free FOVs).
n_fovs_per_well = None

#### Stim treatments — `top` / `middle` / `bottom`

Each StimTreatment fires a single CyanStim pulse at `STIM_FRAME`. The treatment name
is reused as the `stim_location` metadata key consumed by `StimSpotOnCell`.

If you want to run the continuous polar variant instead, skip this cell and use the
**"Polar / Bayesian-optimization variant"** section further down.

In [ ]:
STIM_LOCATIONS = ("top", "middle", "bottom")

stim_phase = [
    StimTreatment(
        treatment_name=loc,                      # reused as stim_location below
        stim_timestep=(STIM_FRAME,),             # single pulse
        stim_exposure_list=100,                  # ms — Moritz used 100 ms
        stim_power=5,                            # %  — start low, calibrate per cell line
        stim_channel_name="CyanStim",
        stim_channel_group=CHANNEL_GROUP,
        stim_channel_device_name="LED",
        stim_channel_power_property_name="Cyan_Level",
        auto_repeat_stim_exposure=True,
    )
    for loc in STIM_LOCATIONS
]

utils.print_stim_exposures_timesteps(stim_phase)

#### Pipeline — segment (nucleus + whole cell), track, ERK-KTR features, place spot

Two segmentations run in parallel on the same imaging frame:

* `"labels"` — **nucleus** mask from a *custom* cellpose model trained on your nuclei stain.
  `FE_ErkKtr` consumes this entry by name (it expands the nucleus into a synthetic
  cytosolic ring internally for the CNR computation).
* `"cells"` — **whole-cell** mask from cellpose `cyto3` on the mScarlet3 channel.
  The stimulator uses this entry (`used_mask="cells"`) so the spot lands on the
  cell shape, not on the small nucleus.

Set `NUC_MODEL_PATH` to your trained nuclei model below; switch to a remote
`SegmentatorImagingServerKit` if you want to offload either segmentation to izbniesen,
same as in `21_cell_migration/cell_migration.ipynb`.

In [ ]:
from faro.segmentation.cellpose import SegmentorCellpose
from faro.tracking.trackpy import TrackerTrackpy
from faro.feature_extraction.erk_ktr import FE_ErkKtr
from faro.feature_extraction.optocheck import OptoCheckFE
from faro.stimulation.spot_on_cell import StimSpotOnCell

# Channels: index 0 -> miRFP (LifeAct, weak nuclear signal), 1 -> mScarlet3 (ERK-KTR).
# Pick the channel each segmentor sees:
NUC_CHANNEL  = 1   # mScarlet3 — ERK-KTR is nucleus-bright when ERK is low; use a dedicated H2B/DAPI channel if you have one
CELL_CHANNEL = 1   # mScarlet3 — whole-cell shape from cytoplasmic+nuclear KTR signal

# Path to your trained nuclei model. Pass a built-in name (e.g. 'nuclei') if you don't have one yet.
NUC_MODEL_PATH = r"Z:\PertzLab\<your-user>\cellpose_models
uclei_custom"   # <- edit me

segmentators = [
    # Nucleus segmentation — keyed as 'labels' so FE_ErkKtr picks it up.
    SegmentationMethod(
        name="labels",
        segmentation_class=SegmentorCellpose(
            model=NUC_MODEL_PATH, is_custom_model=True,
            diameter=30, flow_threshold=0.4, cellprob_threshold=0.0,
            min_size=50,
        ),
        use_channel=NUC_CHANNEL,
        save_tracked=True,
    ),
    # Whole-cell segmentation — keyed as 'cells', consumed by the stimulator.
    SegmentationMethod(
        name="cells",
        segmentation_class=SegmentorCellpose(
            model="cyto3", diameter=80, flow_threshold=1.0, cellprob_threshold=-1.0,
            min_size=200,
        ),
        use_channel=CELL_CHANNEL,
        save_tracked=False,
    ),
]

stimulator = StimSpotOnCell(
    spot_radius=5, height_percentage=0.6, offset=-5,
    clip_to_cell=True, used_mask="cells",         # spot positioned on the whole cell
)
feature_extractor = FE_ErkKtr(used_mask="labels", margin=2, distance=4)   # CNR on the nuclei mask
tracker = TrackerTrackpy()
optocheck = OptoCheckFE(used_mask="labels")

from faro.core.pipeline import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_optocheck=optocheck,
)

#### Quick visual check of the stim-spot placement

Loads a label image from a previous run and previews where the disk would land for each
stim location. Adjust the path before running, or skip this cell.

In [ ]:
# import tifffile
# import matplotlib.pyplot as plt
# labels = tifffile.imread(os.path.join(path, "labels", "000_00000.tiff"))
# fig, axes = plt.subplots(1, len(STIM_LOCATIONS), figsize=(4*len(STIM_LOCATIONS), 4), dpi=120)
# for ax, loc in zip(axes, STIM_LOCATIONS):
#     mask, _ = stimulator.get_stim_mask({"labels": labels}, metadata={"stim_location": loc})
#     ax.imshow(labels > 0, cmap="gray")
#     ax.imshow(np.where(mask, 1, np.nan), cmap="autumn", alpha=0.9)
#     ax.set_title(loc); ax.set_axis_off()
# plt.tight_layout(); plt.show()

### GUI — napari-micromanager

In [ ]:
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None
load_from_file = False

Set ROI (run after the camera has been started once in the GUI):

In [ ]:
if mic.SET_ROI_REQUIRED:
    mic.mmc.clearROI()
    mic.mmc.setROI(mic.ROI_X, mic.ROI_Y, mic.ROI_WIDTH, mic.ROI_HEIGHT)

DMD calibration

In [ ]:
try:
    mm_wdg._core_link.cleanup()
except Exception as e:
    print(e)
import pymmcore_plus

pymmcore_plus.configure_logging(stderr_level="CRITICAL")
mic.calibrate_dmd(verbose=True, exposure=100, radius=4, n_points=15)
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

Skip recalibration and reuse a stored affine matrix:

In [ ]:
# np.save("affine.npy", mic.dmd.affine)
# mic.dmd.affine = np.load("affine.npy")

### Build the acquisition dataframe

Pull FOVs from the napari MDA widget, generate one row per (FOV, timestep), and
spread the three stim treatments across FOVs. The `stim_location` column is what the
`StimSpotOnCell` stimulator reads from event metadata at run time; we copy it from
`treatment_name` so each FOV gets its assigned location.

In [ ]:
fovs = utils.generate_fov_objects(mic, viewer=viewer)

df_acquire = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    channel_optocheck=channel_optocheck,
    optocheck_timepoints=optocheck_timepoints,
    condition=condition,
)
df_acquire = utils.apply_stim_treatments_to_df_acquire(
    df_acquire,
    stim_phase,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)

# Hand the per-treatment location to the stimulator via event metadata.
df_acquire["stim_location"] = df_acquire["treatment_name"]
df_acquire

### Run the experiment

In [ ]:
from faro.core.controller import Controller
from faro.core.writers import OmeZarrWriter
from faro.core.conversion import df_to_events

for _ in range(0, SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600):
    time.sleep(1)

try:
    mm_wdg._core_link.cleanup()
except Exception:
    pass

events = df_to_events(df_acquire)
writer = OmeZarrWriter(storage_path=path)
ctrl = Controller(mic, pipeline, writer=writer)
# stim_mode="previous": frame N's stim uses frame N-1's segmentation and
# fires before frame N's image. With a single pulse at STIM_FRAME=5 this
# matches the Analysis_Exp25 / v2 convention that frame 5 is the first
# post-stim observation. (Use "current" if you want frame 5 to be the
# last pre-stim image instead, like cell_migration.ipynb.)
ctrl.run_experiment(events, stim_mode="previous")
mic.post_experiment()
time.sleep(10)

utils.generate_exp_data_from_tracks(path)

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

### Reconnect / break napari ↔ pymmcore link if needed

In [ ]:
from napari_micromanager._core_link import CoreViewerLink
mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

In [ ]:
# mm_wdg._core_link.cleanup()